# 03 — Limpeza & Camada CURATED

**Objetivo:** aplicar o pré-processamento (escopo, nulos, código 99, outliers de salário, tipagem, decode) e gravar a **camada curated**.

Regras implementadas em `caged_etl/transform.py`.

In [ ]:
import sys, pathlib
# garante que o pacote caged_etl (em ../) seja importável a partir do notebook
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from caged_etl import setup_logging, get_settings
setup_logging()
settings = get_settings()
print('Municípios:', settings.municipios, '| Anos:', settings.ano_inicio, '-', settings.ano_fim)

## 1. Diagnóstico ANTES (no recorte Maceió + Arapiraca)

In [ ]:
import polars as pl
municipios = [str(m) for m in settings.municipios]
lf = (pl.scan_parquet(sorted(settings.raw_dir.glob('caged_mov_*.parquet')))
        .filter(pl.col('municipio').is_in(municipios)))
escopo = lf.collect()
print('linhas no recorte (bruto):', escopo.height)
print('nulos/vazios por coluna crítica:')
for c in ['salario', 'sexo', 'grau_instrucao']:
    n = escopo.filter(pl.col(c).is_null() | (pl.col(c) == '')).height
    print(f'  {c:16s}: {n}')
print('grau de instrução == 99 (ignorado):',
      escopo.filter(pl.col('grau_instrucao') == '99').height)

## 2. Distribuição de salário (antes da limpeza de outliers)

In [ ]:
import matplotlib.pyplot as plt
sal = (escopo.select(pl.col('salario').str.replace_all(r'\\.', '')
                      .str.replace(',', '.').cast(pl.Float64, strict=False))
              .drop_nulls()['salario'])
print(sal.describe())
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].hist(sal.filter(sal <= sal.quantile(0.99)), bins=50)
ax[0].set_title('Salário (<= p99)')
ax[1].boxplot(sal.filter(sal <= sal.quantile(0.99)).to_list(), vert=False)
ax[1].set_title('Boxplot (<= p99)')
plt.tight_layout(); plt.show()

## 3. Construção da camada CURATED

In [ ]:
from caged_etl import transform
curated = transform.build_curated(overwrite=True)
print('linhas curadas:', curated.height)
curated.head()

## 4. Diagnóstico DEPOIS

In [ ]:
print('linhas por município:')
print(curated.group_by('municipio_desc').len())
print('\nnulos críticos restantes (esperado 0):')
for c in ['salario', 'sexo', 'grau_instrucao']:
    print(f'  {c:16s}: {curated.filter(pl.col(c).is_null()).height}')
print('\ngrau 99 restante (esperado 0):',
      curated.filter(pl.col('grau_instrucao') == '99').height)
print('salário mín/máx:', curated['salario'].min(), curated['salario'].max())

## ✅ Decisões de limpeza documentadas
- **Escopo**: somente Maceió/Arapiraca, 2021–2025.
- **Nulos críticos** (`salario`, `sexo`, `grau_instrucao`): linhas removidas.
- **Código 99** em grau de instrução: removido (não distorcer médias).
- **Outliers de salário**: `<= 0` e `> R$ 1.000.000` removidos (teto refinável por IQR).
- **Tipagem**: salário `float` (vírgula→ponto), idade/saldo inteiros.
- **Decode**: colunas `*_desc` adicionadas para leitura.